## Langgraph sample code

In [6]:
import os
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

### 1) Load API key from .env

In [7]:
# 1) Load API key from .env
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Missing GEMINI_API_KEY in .env")

In [ ]:
# 2) Use a valid Gemini model
# Common valid names: gemini-2.5-flash, gemini-2.5-flash-lite
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0
)

# 3) Define tools
@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """Divide two integers."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

tools = [add, multiply, divide]
model_with_tools = model.bind_tools(tools)

# 4) Test the model with a tool-capable prompt
response = model_with_tools.invoke(
    "Use the tools to calculate (3 + 5) * 2 and then divide by 4."
)

print(response)

content='' additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"b": 5, "a": 3}'}, '__gemini_function_call_thought_signatures__': {'3c902a7e-0ea1-4696-8d2a-1f878853655e': 'CqQDARFNMg9aJ0VwYey4HaCXh59GEMa3lflDXnh84pq5phXQmTOQV5CB/p8ybof+u7Os8SsffHbSEG0qA92cYu6YEUn6XueG3qR6/j+pTbP5Jm+kBau3pcL2bAkJdTSuK4yNuF0NC8R+ofULmECT77JtwQYQPVGYNH9NzbNTFbKCN7A+ZauZvf+lcnzb/mNLGd2ySsY9rbO3YdAWwKsvExuH7+xsTVmIhilWX8refEyCwK3qZ6BVIuTaPP7OELd/gJjQ0hqZcjfMRJ0Ic4K5xMlkplNG4IQf9ysFgBxISbsgDDefPwIKX5ty6Ku+cNlrl82ySW1hJHB8osXQXjLn1Cd/L0hWbz+cdLU9UIDmjkdofrBaL+PL0d3IKPbtrbYRYxeAHao4ztyx3Vz6oBm313BvcuD3A+XzcsaX4mW3WCC5BkeIvrW/wJ/oGOjRyppgBRRDC/6KDdLvEPGzwA0fXTDZoo4byoZ7IFLnRjkTecxCOL1qOTLoGAzfNvKQOBw27xwAyqXsL+OWd4Ees2lCOzH9vQXpPFYizR9puJC6dPNoQiDBi+pr'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a01383-0cf3-7f20-96c5-39c0c9638155-0' tool_calls=[{'name': 'add', 'args': {'b': 5, 'a': 3}, 'id': 

### 2. Define state
#### The graph’s state is used to store the messages and the number of LLM calls.

In [9]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator


class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int


### 3. Define model node
#### The model node is used to call the LLM and decide whether to call a tool or not.

In [ ]:
from langchain.messages import SystemMessage
def llm_call(state: dict):
    """LLM decides whether to call a tool or not"""
    return {
        "messages": [
            model_with_tools.invoke(
                [
                    SystemMessage(
                        content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
                    )
                ]
                + state["messages"] # can be used as "CONTEXT
            )
        ],
        "llm_calls": state.get('llm_calls', 0) + 1
    }


### 4. Define tool node
#### The tool node is used to call the tools and return the results.

In [11]:
from langchain.messages import ToolMessage
def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

### 5. Define end logic
#### The conditional edge function is used to route to the tool node or end based upon whether the LLM made a tool call.

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END

def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]

    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tool_node"

    # Otherwise, we stop (reply to the user)
    return END

6. Build and compile the agent
The agent is built using the StateGraph class and compiled using the compile method.

In [ ]:
# Build workflow
agent_builder = StateGraph(MessagesState)

# Add nodes
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# Add edges to connect nodes
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]
)
agent_builder.add_edge("tool_node", "llm_call")

# Compile the agent
agent = agent_builder.compile()

In [ ]:
# Show the agent
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
# Invoke
from langchain.messages import HumanMessage
messages = [HumanMessage(content="Add 3 and 4.")]
messages = agent.invoke({"messages": messages})
for m in messages["messages"]:
    m.pretty_print()